In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("Week6SparkAssignment") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("Downloads/source_dataset.csv")

In [4]:
df.show(5)
# show top 5 rows of the dataset 

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|product_id| old_name|   category|base_price|quantity|   status|region|priority|user_id|  price| amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13|1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47|8150.82|
|    P00003|Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33|3873.31|
|    P00004|    Novel|Electronics|    594.61|       1|Completed| South|  Medium|  U0850| 594.61| 594.61|
|    P00005|  T-Shirt|   Clothing|   1075.16|       2|Cancelled| North|    High|  U0246|1075.16|2150.32|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
only showing top 5 rows


In [5]:
df.printSchema()
# shows the datatypes of the columns

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)



In [6]:
df.count()
# dataset has 5000 rows

5000

# Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

In [7]:
#Answer:
#In Spark architecture,
#The Driver is the main process that controls the application. It creates the execution plan and sends tasks for execution.
#The Cluster Manager is responsible for managing resources such as CPU and memory and allocating them to the Spark application.
#The Executors are responsible for actually executing the tasks given by the Driver. They process the data and can also store intermediate results.

#driver -> cluster manager -> Executors -> data processing (this is the flow in the spark architecturee)

# Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

In [8]:
#Answer
#Spark follows lazy evaluation which means transformations are not executed immediately.Spark does not immediately process all the data after the commands.
#It keeps track of the transformations.Execution happens when an action is called.This improves performance because Spark gets a chance to look at all the
#transformations together and create an optimized execution plan before processing the data.

electronics_df = df.filter(col("category") == "Electronics")

result_df = electronics_df.select(
    "product_id",
    "price"
)
result_df.show(5)

+----------+-------+
|product_id|  price|
+----------+-------+
|    P00004| 594.61|
|    P00014| 106.21|
|    P00016|2656.39|
|    P00025|3493.32|
|    P00026|1293.29|
+----------+-------+
only showing top 5 rows


# Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [9]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("Downloads/source_dataset.csv")

df.show(5)
df.printSchema()
#Here header tells Spark that the first row contains the column names
#while inferSchema allows Spark to automatically identify the data types of the columns.

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|product_id| old_name|   category|base_price|quantity|   status|region|priority|user_id|  price| amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13|1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47|8150.82|
|    P00003|Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33|3873.31|
|    P00004|    Novel|Electronics|    594.61|       1|Completed| South|  Medium|  U0850| 594.61| 594.61|
|    P00005|  T-Shirt|   Clothing|   1075.16|       2|Cancelled| North|    High|  U0246|1075.16|2150.32|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
only showing top 5 rows
root
 |-- product_id: string (n

# Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

In [10]:
#CSV and Parquet store data differently.
#CSV is a row-based text format, whereas Parquet is a columnar file format.
#For example, if I have 20 columns but only need product_id and price,
#Parquet can read the required columns more efficiently instead of processing unnecessary data.
#Parquet also provides better compression and stores schema information.
#Because of this, Parquet is generally more suitable than CSV when processing large analytical datasets in Spark.

# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [11]:
df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
).show(10)
#filtered the dataset for products belonging to the Electronics category and selected only the required columns.

+----------+-------+
|product_id|  price|
+----------+-------+
|    P00004| 594.61|
|    P00014| 106.21|
|    P00016|2656.39|
|    P00025|3493.32|
|    P00026|1293.29|
|    P00029|1412.92|
|    P00030|2648.95|
|    P00034|1871.88|
|    P00059| 929.09|
|    P00064|4470.18|
+----------+-------+
only showing top 10 rows


# Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [12]:
df_revised = df \
    .withColumnRenamed("old_name", "new_name") \
    .withColumn("price", col("price").cast("double"))
df_revised.printSchema()
df_revised.show(5)
# renamed the old_name column to new_name ,also converted the price column to the double data type from string.

root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|product_id| new_name|   category|base_price|quantity|   status|region|priority|user_id|  price| amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13|1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47|8150.82|
|    P00003|Face Wash|      Books|

# Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

In [13]:
#Answer
#Spark keeps track of the transformations performed on the data using a Lineage Graph or DAG (Directed Acyclic Graph).
#If a worker fails and some data is lost, Spark can use this lineage information to determine how that data was created.
#Instead of restarting the complete job, Spark can recompute the lost partitions using the previous transformations.This provides fault tolerance.

# Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [37]:
df_orders = df
df_orders.filter(
    (col("status") == "Completed") & (col("amount") > 1000)
).show(10)

+----------+-------------+-----------+----------+--------+---------+------+--------+-------+-------+--------+
|product_id|     old_name|   category|base_price|quantity|   status|region|priority|user_id|  price|  amount|
+----------+-------------+-----------+----------+--------+---------+------+--------+-------+-------+--------+
|    P00001|        Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13| 1080.26|
|    P00002|        Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47| 8150.82|
|    P00003|    Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33| 3873.31|
|    P00009|   Headphones|     Beauty|    792.09|       7|Completed|  West|     Low|  U0645| 792.09| 5544.63|
|    P00010|      T-Shirt|     Sports|   4611.08|       4|Completed|  West|  Medium|  U1129|4611.08|18444.32|
|    P00013|      T-Shirt|     Beauty|   3093.54|       4|Completed|  West|  Medium|  U0630|3093.54|12374.16|
|    P0001

# Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory

In [ ]:
parquet_df.filter(
    col("category") == "Electronics"
)
#Predicate Pushdown is an optimization that allows Spark to apply filtering conditions closer to the data source.
#When working with formats such as Parquet, Spark can use file metadata to avoid reading irrelevant data where possible.
#means less data needs to be read and processed, which can improve both execution speed and memory efficiency.

# Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [14]:
df_with_tax = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)
df_with_tax.select(
    "product_id",
    "base_price",
    "final_price"
).show(10)
# new column is created final_price, it is calculated by multiplying base_price with 1.18

+----------+----------+------------------+
|product_id|base_price|       final_price|
+----------+----------+------------------+
|    P00001|    540.13|          637.3534|
|    P00002|   1358.47|         1602.9946|
|    P00003|    553.33|          652.9294|
|    P00004|    594.61| 701.6397999999999|
|    P00005|   1075.16|1268.6888000000001|
|    P00006|   4999.26|         5899.1268|
|    P00007|     110.7|           130.626|
|    P00008|    3229.5|           3810.81|
|    P00009|    792.09|          934.6662|
|    P00010|   4611.08|5441.0743999999995|
+----------+----------+------------------+
only showing top 10 rows


# Q11: What is the difference between Transformations and Actions? Provide two examples of each.

In [15]:
# Answer
# In Spark, Transformations are operations that create a new DataFrame from an existing DataFrame.
#They are lazily evaluated, which means Spark does not execute them immediately.
#The execution happens only when an action is called.

#transformation
# Example 1: Filter
df.filter(col("price") > 1000)
# Example 2: Select
df.select("product_id", "price")

#action
# Example 1: Show
df.show(5)
# Example 2: Count
df.count()

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|product_id| old_name|   category|base_price|quantity|   status|region|priority|user_id|  price| amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13|1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47|8150.82|
|    P00003|Face Wash|      Books|    553.33|       7|Completed|  West|     Low|  U0207| 553.33|3873.31|
|    P00004|    Novel|Electronics|    594.61|       1|Completed| South|  Medium|  U0850| 594.61| 594.61|
|    P00005|  T-Shirt|   Clothing|   1075.16|       2|Cancelled| North|    High|  U0246|1075.16|2150.32|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+-------+
only showing top 5 rows


5000

# Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

In [ ]:
#Answer
#In Client Mode, the Spark Driver runs on the machine from where the Spark application is submitted.
#This is useful during development because the output and errors can be viewed directly.
#In Cluster Mode, the Driver runs inside the cluster itself.
#Cluster Mode is generally more suitable for production workloads because the application does not depend on the client machine staying connected.

# Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [16]:
df.filter(
    (col("region") == "North") | (col("priority") == "High")
).show(10)

+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+--------+
|product_id| old_name|   category|base_price|quantity|   status|region|priority|user_id|  price|  amount|
+----------+---------+-----------+----------+--------+---------+------+--------+-------+-------+--------+
|    P00001|    Novel|       Home|    540.13|       2|Completed| North|    High|  U1405| 540.13| 1080.26|
|    P00002|    Mixer|   Clothing|   1358.47|       6|Completed| North|  Medium|  U0669|1358.47| 8150.82|
|    P00005|  T-Shirt|   Clothing|   1075.16|       2|Cancelled| North|    High|  U0246|1075.16| 2150.32|
|    P00006|Face Wash|     Sports|   4999.26|       2|  Pending| North|    High|  U0884|4999.26| 9998.52|
|    P00007|Face Wash|     Beauty|     110.7|       6|Completed|  East|    High|  U0083|  110.7|   664.2|
|    P00008|    Novel|     Sports|    3229.5|       4| Returned|  East|    High|  U0317| 3229.5| 12918.0|
|    P00011|    Novel|       Home|     558.8| 

# Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 

In [ ]:
#Answer
#.show(5) is safer because it only displays the first 5 rows, which is enough to quickly understand what the data looks like.
#On the other hand, .collect() tries to bring the entire dataset into the Driver’s memory.
#If the dataset is several terabytes in size, it can consume all the available memory and may even crash the Spark application.
#So when we only want to explore or check the dataset, we would use .show(5) instead of loading everything with .collect().

# wide transformation

In [17]:
#While working with the dataset used groupBy() to group the records based on category.
#Understood that groupBy() is a wide transformation because Spark may need to move data between different partitions
#so that similar records can be processed together. This movement of data is called a shuffle.
#Since shuffling involves transferring data between executors and may also involve disk I/O, it can increase the processing time.
#Therefore, while working with large datasets, shuffle-heavy operations should be used carefully and optimized wherever possible.
df.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|       Home|  822|
|     Sports|  849|
|Electronics|  811|
|   Clothing|  883|
|      Books|  827|
|     Beauty|  808|
+-----------+-----+

